# Student Performance Prediction: Complete Machine Learning Workflow

This notebook performs a complete machine learning analysis of the **Student Habits and Performance** dataset.

## Objectives
1. Perform Exploratory Data Analysis (EDA).
2. Identify numerical and categorical variables.
3. Examine distributions, relationships, correlations, missing values, and potential outliers.
4. Identify factors related to `exam_score`.
5. Build and evaluate a **Linear Regression** model.
6. Convert `exam_score` into a **Pass/Fail** classification target using a 50-mark threshold.
7. Train and evaluate a classification model.
8. Compare training and testing performance for possible overfitting or underfitting.
9. Summarize the most important findings and provide 5 meaningful insights.


In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report
)

import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)


In [ ]:
# Load the dataset
file_path = "Day18_19_student_habits_performance.csv"

df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)
display(df.head())


## 1. Initial Dataset Inspection

In [ ]:
# Dataset information and descriptive statistics
print("Dataset information:")
df.info()

print("\nDescriptive statistics:")
display(df.describe(include="all").T)

print("\nColumn names:")
print(df.columns.tolist())


In [ ]:
# Identify numerical and categorical variables
numerical_cols = df.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df.select_dtypes(exclude=np.number).columns.tolist()

print("Numerical variables:")
print(numerical_cols)

print("\nCategorical variables:")
print(categorical_cols)


## 2. Missing Values Analysis

In [ ]:
# Missing values
missing_values = df.isnull().sum().sort_values(ascending=False)
missing_percent = (missing_values / len(df) * 100).round(2)

missing_summary = pd.DataFrame({
    "Missing Values": missing_values,
    "Percentage (%)": missing_percent
})

display(missing_summary[missing_summary["Missing Values"] > 0])

print("Total missing values:", df.isnull().sum().sum())


In [ ]:
# Visualize missing values
plt.figure(figsize=(10, 5))
sns.heatmap(df.isnull(), cbar=False, cmap="viridis")
plt.title("Missing Values Heatmap")
plt.xlabel("Variables")
plt.ylabel("Observations")
plt.show()


## 3. Target Variable Analysis: `exam_score`

In [ ]:
# Distribution of exam_score
plt.figure(figsize=(10, 5))
sns.histplot(df["exam_score"], kde=True, bins=30)
plt.title("Distribution of Exam Scores")
plt.xlabel("Exam Score")
plt.show()

print(df["exam_score"].describe())


In [ ]:
# Boxplot for exam_score
plt.figure(figsize=(8, 4))
sns.boxplot(x=df["exam_score"])
plt.title("Boxplot of Exam Scores")
plt.show()


## 4. Numerical Variable Distributions

In [ ]:
# Histograms for numerical variables
df[numerical_cols].hist(figsize=(16, 12), bins=25, edgecolor="black")
plt.suptitle("Distributions of Numerical Variables", fontsize=16)
plt.tight_layout()
plt.show()


## 5. Categorical Variable Analysis

In [ ]:
# Value counts and count plots for categorical variables
for col in categorical_cols:
    print(f"\n--- {col} ---")
    display(df[col].value_counts(dropna=False).to_frame("Count"))

    plt.figure(figsize=(7, 4))
    sns.countplot(data=df, x=col, order=df[col].value_counts().index)
    plt.title(f"Distribution of {col}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


## 6. Relationship Between Variables and `exam_score`

In [ ]:
# Scatter plots for important numerical predictors vs exam_score
important_numeric_features = [
    "study_hours_per_day",
    "attendance_percentage",
    "sleep_hours",
    "social_media_hours",
    "netflix_hours",
    "exercise_frequency",
    "mental_health_rating"
]

for col in important_numeric_features:
    if col in df.columns:
        plt.figure(figsize=(7, 4))
        sns.scatterplot(data=df, x=col, y="exam_score", alpha=0.6)
        sns.regplot(data=df, x=col, y="exam_score", scatter=False)
        plt.title(f"{col} vs Exam Score")
        plt.tight_layout()
        plt.show()


In [ ]:
# Average exam score by categorical variables
categorical_features = [
    col for col in categorical_cols
    if col not in ["student_id"]
]

for col in categorical_features:
    summary = df.groupby(col, dropna=False)["exam_score"].agg(["mean", "median", "count"]).sort_values("mean", ascending=False)
    print(f"\nAverage Exam Score by {col}:")
    display(summary)

    plt.figure(figsize=(8, 4))
    sns.boxplot(data=df, x=col, y="exam_score")
    plt.title(f"Exam Score by {col}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


## 7. Correlation Analysis

In [ ]:
# Correlation matrix for numerical variables
correlation_matrix = df[numerical_cols].corr()

plt.figure(figsize=(12, 8))
sns.heatmap(correlation_matrix, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title("Correlation Matrix")
plt.show()

# Correlations with exam_score
exam_correlations = correlation_matrix["exam_score"].sort_values(ascending=False)

print("Correlation of numerical variables with exam_score:")
display(exam_correlations.to_frame("Correlation with exam_score"))


## 8. Potential Outlier Detection

In [ ]:
# IQR-based outlier summary
outlier_summary = []

for col in numerical_cols:
    if col == "exam_score":
        continue

    series = df[col].dropna()
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = ((series < lower_bound) | (series > upper_bound)).sum()

    outlier_summary.append({
        "Variable": col,
        "Outlier Count": outliers,
        "Outlier Percentage": round(outliers / len(series) * 100, 2),
        "Lower Bound": round(lower_bound, 2),
        "Upper Bound": round(upper_bound, 2)
    })

outlier_df = pd.DataFrame(outlier_summary).sort_values("Outlier Count", ascending=False)
display(outlier_df)


In [ ]:
# Boxplots for numerical variables
plt.figure(figsize=(16, 10))
df[numerical_cols].plot(kind="box", subplots=True, layout=(3, 3), figsize=(16, 10))
plt.tight_layout()
plt.show()


## EDA Interpretation

The next cell automatically identifies the strongest numerical relationships with `exam_score`. Correlation should be interpreted as association, not causation. The model will use multiple features simultaneously, allowing us to evaluate predictive performance rather than relying only on individual correlations.


In [ ]:
# Automatically identify strongest numerical relationships
relationship_summary = exam_correlations.drop("exam_score").sort_values(key=lambda x: x.abs(), ascending=False)

print("Strongest numerical relationships with exam_score:")
display(relationship_summary.to_frame("Correlation").head(10))

print("\nTop positive relationships:")
display(relationship_summary[relationship_summary > 0].head(5).to_frame("Correlation"))

print("\nTop negative relationships:")
display(relationship_summary[relationship_summary < 0].head(5).to_frame("Correlation"))


# PART A: REGRESSION MODELING

## 9. Data Preparation for Regression

- Target variable: `exam_score`
- `student_id` is excluded because it is an identifier rather than a meaningful predictive feature.
- Missing numerical values are imputed using the median.
- Missing categorical values are imputed using the most frequent category.
- Categorical variables are one-hot encoded.


In [ ]:
# Prepare features and regression target
X = df.drop(columns=["exam_score", "student_id"], errors="ignore")
y = df["exam_score"]

numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

print("Regression features:")
print(X.columns.tolist())

print("\nNumerical features:", numeric_features)
print("\nCategorical features:", categorical_features)


In [ ]:
# Preprocessing pipelines
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)


In [ ]:
# Train-test split for regression
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

print("Training set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)


## 10. Train the Linear Regression Model

In [ ]:
# Linear Regression pipeline
regression_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

regression_model.fit(X_train, y_train)

# Predictions
y_train_pred = regression_model.predict(X_train)
y_test_pred = regression_model.predict(X_test)


## 11. Regression Model Evaluation

In [ ]:
# Regression evaluation function
def regression_metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "MSE": mean_squared_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R² Score": r2_score(y_true, y_pred)
    }

train_reg_metrics = regression_metrics(y_train, y_train_pred)
test_reg_metrics = regression_metrics(y_test, y_test_pred)

regression_results = pd.DataFrame(
    [train_reg_metrics, test_reg_metrics],
    index=["Training Data", "Testing Data"]
)

display(regression_results)


In [ ]:
# Actual vs predicted values
plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_test_pred, alpha=0.7)
min_value = min(y_test.min(), y_test_pred.min())
max_value = max(y_test.max(), y_test_pred.max())
plt.plot([min_value, max_value], [min_value, max_value], "r--")
plt.xlabel("Actual Exam Score")
plt.ylabel("Predicted Exam Score")
plt.title("Actual vs Predicted Exam Scores")
plt.show()


In [ ]:
# Residual analysis
residuals = y_test - y_test_pred

plt.figure(figsize=(8, 5))
sns.scatterplot(x=y_test_pred, y=residuals)
plt.axhline(y=0, color="red", linestyle="--")
plt.xlabel("Predicted Exam Score")
plt.ylabel("Residual")
plt.title("Residual Plot")
plt.show()

plt.figure(figsize=(8, 4))
sns.histplot(residuals, kde=True)
plt.title("Distribution of Regression Residuals")
plt.xlabel("Residual")
plt.show()


## 12. Linear Regression Feature Effects

In [ ]:
# Extract regression coefficients after preprocessing
feature_names = regression_model.named_steps["preprocessor"].get_feature_names_out()
coefficients = regression_model.named_steps["model"].coef_

coef_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients,
    "Absolute Coefficient": np.abs(coefficients)
}).sort_values("Absolute Coefficient", ascending=False)

print("Top 15 features by absolute coefficient magnitude:")
display(coef_df.head(15))

plt.figure(figsize=(10, 7))
top_coef = coef_df.head(15).sort_values("Coefficient")
plt.barh(top_coef["Feature"], top_coef["Coefficient"])
plt.xlabel("Regression Coefficient")
plt.title("Top 15 Linear Regression Feature Effects")
plt.tight_layout()
plt.show()


# PART B: CLASSIFICATION MODELING

## 13. Create Pass/Fail Target

A student is considered:
- **Pass (1):** `exam_score >= 50`
- **Fail (0):** `exam_score < 50`


In [ ]:
# Create Pass/Fail classification target
y_class = (df["exam_score"] >= 50).astype(int)

df_classification = df.copy()
df_classification["pass_fail"] = y_class.map({0: "Fail", 1: "Pass"})

print("Class distribution:")
display(df_classification["pass_fail"].value_counts())

plt.figure(figsize=(6, 4))
sns.countplot(data=df_classification, x="pass_fail")
plt.title("Pass/Fail Class Distribution")
plt.show()


In [ ]:
# Classification features
X_class = df.drop(columns=["exam_score", "student_id"], errors="ignore")

# Stratified split preserves the Pass/Fail class balance
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_class, y_class,
    test_size=0.20,
    random_state=42,
    stratify=y_class
)

print("Training class distribution:")
print(yc_train.value_counts(normalize=True).rename({0: "Fail", 1: "Pass"}))

print("\nTesting class distribution:")
print(yc_test.value_counts(normalize=True).rename({0: "Fail", 1: "Pass"}))


## 14. Train Classification Model: Logistic Regression

In [ ]:
# Logistic Regression model
classification_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=2000, random_state=42))
])

classification_model.fit(Xc_train, yc_train)

# Predictions
yc_train_pred = classification_model.predict(Xc_train)
yc_test_pred = classification_model.predict(Xc_test)


## 15. Classification Model Evaluation

In [ ]:
# Classification metrics function
def classification_metrics(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1-score": f1_score(y_true, y_pred, zero_division=0)
    }

train_class_metrics = classification_metrics(yc_train, yc_train_pred)
test_class_metrics = classification_metrics(yc_test, yc_test_pred)

classification_results = pd.DataFrame(
    [train_class_metrics, test_class_metrics],
    index=["Training Data", "Testing Data"]
)

display(classification_results)


In [ ]:
# Confusion matrix
cm = confusion_matrix(yc_test, yc_test_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Fail", "Pass"]
)
disp.plot()
plt.title("Confusion Matrix - Test Data")
plt.show()

print("Classification Report:")
print(classification_report(
    yc_test,
    yc_test_pred,
    target_names=["Fail", "Pass"],
    zero_division=0
))


## 16. Classification Feature Importance

In [ ]:
# Logistic regression coefficients
class_feature_names = classification_model.named_steps["preprocessor"].get_feature_names_out()
class_coefficients = classification_model.named_steps["model"].coef_[0]

class_coef_df = pd.DataFrame({
    "Feature": class_feature_names,
    "Coefficient": class_coefficients,
    "Absolute Coefficient": np.abs(class_coefficients)
}).sort_values("Absolute Coefficient", ascending=False)

print("Top 15 features influencing Pass/Fail prediction:")
display(class_coef_df.head(15))

plt.figure(figsize=(10, 7))
top_class_coef = class_coef_df.head(15).sort_values("Coefficient")
plt.barh(top_class_coef["Feature"], top_class_coef["Coefficient"])
plt.xlabel("Logistic Regression Coefficient")
plt.title("Top 15 Classification Feature Effects")
plt.tight_layout()
plt.show()


# PART C: MODEL COMPARISON AND CONCLUSIONS

## 17. Compare Training and Testing Performance

In [ ]:
# Compare regression and classification training/testing performance
print("REGRESSION PERFORMANCE")
display(regression_results)

print("\nCLASSIFICATION PERFORMANCE")
display(classification_results)


In [ ]:
# Automatic generalization-gap interpretation
reg_r2_gap = train_reg_metrics["R² Score"] - test_reg_metrics["R² Score"]
class_acc_gap = train_class_metrics["Accuracy"] - test_class_metrics["Accuracy"]

print("Regression R² gap (Train - Test):", round(reg_r2_gap, 4))
print("Classification Accuracy gap (Train - Test):", round(class_acc_gap, 4))

print("\nInterpretation:")
if reg_r2_gap > 0.10:
    print("- Regression: Noticeable train-test gap may indicate overfitting.")
elif test_reg_metrics["R² Score"] < 0.10 and train_reg_metrics["R² Score"] < 0.10:
    print("- Regression: Low performance on both train and test data may indicate underfitting.")
else:
    print("- Regression: Training and testing performance are relatively consistent.")

if class_acc_gap > 0.10:
    print("- Classification: Noticeable train-test accuracy gap may indicate overfitting.")
elif train_class_metrics["Accuracy"] < 0.60 and test_class_metrics["Accuracy"] < 0.60:
    print("- Classification: Low accuracy on both datasets may indicate underfitting.")
else:
    print("- Classification: Training and testing accuracy are relatively consistent.")


## 18. Automatic Summary of Important Findings

In [ ]:
# Summarize important EDA and model findings
top_corr = relationship_summary.head(5)

print("1. Strongest numerical relationships with exam_score:")
for feature, corr in top_corr.items():
    direction = "positive" if corr > 0 else "negative"
    print(f"   - {feature}: {corr:.3f} ({direction} relationship)")

print("\n2. Regression test performance:")
for metric, value in test_reg_metrics.items():
    print(f"   - {metric}: {value:.4f}")

print("\n3. Classification test performance:")
for metric, value in test_class_metrics.items():
    print(f"   - {metric}: {value:.4f}")

print("\n4. Most influential regression features:")
display(coef_df[["Feature", "Coefficient"]].head(5))

print("\n5. Most influential classification features:")
display(class_coef_df[["Feature", "Coefficient"]].head(5))


## 19. Five Meaningful Insights About Student Performance

In [ ]:
# Generate five data-supported insights
top_feature = relationship_summary.index[0]
top_corr_value = relationship_summary.iloc[0]

positive_features = relationship_summary[relationship_summary > 0]
negative_features = relationship_summary[relationship_summary < 0]

print("INSIGHT 1:")
print(f"The strongest numerical association with exam_score is {top_feature} "
      f"with a correlation of {top_corr_value:.3f}. This indicates it is one of the most relevant numerical factors in the dataset.")

print("\nINSIGHT 2:")
if len(positive_features) > 0:
    print(f"Variables with positive relationships to exam_score include: {', '.join(positive_features.head(3).index.tolist())}. "
          "Higher values in these variables tend to be associated with higher exam scores.")

print("\nINSIGHT 3:")
if len(negative_features) > 0:
    print(f"Variables with negative relationships to exam_score include: {', '.join(negative_features.head(3).index.tolist())}. "
          "Higher values in these variables tend to be associated with lower exam scores.")

print("\nINSIGHT 4:")
print(f"The Linear Regression model achieved a test R² score of {test_reg_metrics['R² Score']:.3f} "
      f"and a test RMSE of {test_reg_metrics['RMSE']:.3f}. This shows how accurately the selected student characteristics predict exam scores.")

print("\nINSIGHT 5:")
print(f"The Pass/Fail classification model achieved test accuracy of {test_class_metrics['Accuracy']:.3f}, "
      f"precision of {test_class_metrics['Precision']:.3f}, recall of {test_class_metrics['Recall']:.3f}, "
      f"and F1-score of {test_class_metrics['F1-score']:.3f}. These metrics show the model's ability to identify students who pass the 50-mark threshold.")


## 20. Final Conclusion

This notebook completed the full machine learning workflow:

- The dataset was inspected and explored using descriptive statistics and visualizations.
- Numerical and categorical variables were identified.
- Missing values and potential outliers were examined.
- Relationships and correlations with `exam_score` were analyzed.
- A Linear Regression model was trained to predict the continuous exam score.
- Regression performance was evaluated using **MAE, MSE, RMSE, and R²**.
- `exam_score` was converted into a Pass/Fail target using **50 marks** as the threshold.
- A Logistic Regression classifier was trained and evaluated using a **confusion matrix, accuracy, precision, recall, and F1-score**.
- Training and testing performance were compared to assess possible overfitting or underfitting.
- The final insights are generated directly from the EDA and model results.

### Note
Run the notebook from top to bottom. The final findings are data-driven and will automatically reflect the actual results obtained from the dataset.
